# NBA March Madness Analysis

Brooke Rios

ADSP 31017

Winter 2026

---

## Exploratory Data Analysis

### Data Processing

In [1]:
%matplotlib inline
%config HistoryManager.enabled = False

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

#### Teams Table

In [2]:
import requests
import pandas as pd
import time

API_KEY = "7f10-3c2730"
BASE_URL = "https://api3.natst.at"
SPORT_CODE = "MBB"
SEASON = "2024"

# ----------------------------
# Paginated fetch
# ----------------------------
def fetch_paginated(endpoint, range_param=None):
    """
    Fetch all records from a paginated NatStat endpoint.
    Returns a list of records.
    """
    all_records = []
    offset = 0

    while True:
        if range_param:
            url = f"{BASE_URL}/{API_KEY}/{endpoint}/{SPORT_CODE}/{range_param}/{offset}"
        else:
            url = f"{BASE_URL}/{API_KEY}/{endpoint}/{SPORT_CODE}/{offset}"

        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        # Identify the records
        if endpoint == "teams":
            records = data.get("teams", {})
        elif endpoint == "games":
            records = data.get("games", {})
        elif endpoint == "teamperfs" or endpoint == "performances":
            records = data.get("performances", {})
        elif endpoint == "elo":
            records = data.get("elo", {})
        else:
            records = data.get("stats", {})

        if not records:
            break

        # Convert dict to list of dicts
        for key, value in records.items():
            all_records.append(value)

        # Pagination: stop if <100 records returned
        if len(records) < 100:
            break

        offset += 100
        time.sleep(0.3)  # small delay to avoid rate limits

    return all_records

# ----------------------------
# Load Teams
# ----------------------------
teams_records = fetch_paginated("teams", SEASON)
teams_df = pd.json_normalize(teams_records)

print("Teams DF shape:", teams_df.shape)
teams_df.head()

Teams DF shape: (362, 3)


,code,name,location
0,ACU,Abilene Christian Wildcats,NaN
1,AFA,Air Force Falcons,NaN
2,AKR,Akron Zips,NaN
3,ALA,Alabama Crimson Tide,NaN
4,ALAM,Alabama A&amp;M Bulldogs,NaN


#### Games Table

In [3]:
games_records = fetch_paginated("games", SEASON)
games_df = pd.json_normalize(games_records)

print("Games DF shape:", games_df.shape)
games_df.head()

Games DF shape: (6219, 17)


,id,visitor,visitor-code,score-vis,home,home-code,score-home,gamestatus,overtime,winner-code,loser-code,gameday,gameno,venue,venue-code,neutral,league
0,1259345,Purdue Boilermakers,PUR,60,Connecticut Huskies,CONN,75,Final,N,CONN,PUR,2024-04-08,1,State Farm Stadium,680,Y,NaN
1,1259340,North Carolina State Wolfpack,NCSU,50,Purdue Boilermakers,PUR,63,Final,N,PUR,NCSU,2024-04-06,1,State Farm Stadium,680,Y,NaN
2,1259338,Alabama Crimson Tide,ALA,72,Connecticut Huskies,CONN,86,Final,N,CONN,ALA,2024-04-06,1,State Farm Stadium,680,Y,NaN
3,1259343,Indiana State Sycamores,INST,77,Seton Hall Pirates,SETON,79,Final,N,SETON,INST,2024-04-04,NaN,Prudential Center,900,Y,NaN
4,1259330,Georgia Bulldogs,UGA,67,Seton Hall Pirates,SETON,84,Final,N,SETON,UGA,2024-04-02,1,Gainsbridge Fieldhouse,264,Y,NaN


#### Team Stats Table

In [4]:
import requests
import pandas as pd

# ================================
# Configuration
# ================================
API_KEY = "7f10-3c2730"
BASE_URL = "https://api3.natst.at"
SPORT_CODE = "MBB"  # NCAA Men's Division I Basketball
SEASON = "2024"

# ================================
# Fetch glossary and get team stat codes
# ================================
glossary_url = f"{BASE_URL}/{API_KEY}/glossary/{SPORT_CODE}"
glossary = requests.get(glossary_url).json()

# Extract valid team stat codes
team_stat_codes = [
    code for code, info in glossary.get("stats", {}).items()
    if info.get("scope") == "team"
]

print("Sample team stat codes:", team_stat_codes[:10])

# ================================
# Function to fetch one team stat
# ================================
def fetch_team_stat(stat_code, season=SEASON):
    """
    Fetch a single team-level stat from NatStat.
    Returns a DataFrame with columns: team_code, team_name, stat_code value
    """
    url = f"{BASE_URL}/{API_KEY}/stats/{SPORT_CODE}/team,{stat_code},{season}"
    response = requests.get(url)

    if response.status_code != 200:
        print(f"Failed to fetch {stat_code}: {response.status_code}")
        return pd.DataFrame()

    data = response.json()
    if not data.get("stats"):
        print(f"No data for stat: {stat_code}")
        return pd.DataFrame()

    records = []
    for team_code, team_data in data["stats"].items():
        records.append({
            "team_code": team_code,
            "team_name": team_data.get("name"),
            stat_code: team_data.get("value")
        })

    return pd.DataFrame(records)

# ================================
# Fetch multiple team stats
# ================================
# Dictionary to hold separate DataFrames per stat
team_stats_dfs = {}

# Example: fetch first 10 stats for testing
for stat_code in team_stat_codes[:10]:
    print(f"Fetching {stat_code}...")
    df = fetch_team_stat(stat_code)
    if not df.empty:
        team_stats_dfs[stat_code] = df
    else:
        print(f"Skipping {stat_code}, no data")

# Inspect keys
print("Fetched team stats DataFrames:", list(team_stats_dfs.keys()))

Sample team stat codes: []
Fetched team stats DataFrames: []


In [5]:
import requests
API_KEY = "7f10-3c2730"
BASE_URL = "https://api3.natst.at"
SPORT_CODE = "MBB"

glossary_url = f"{BASE_URL}/{API_KEY}/glossary/{SPORT_CODE}"
glossary = requests.get(glossary_url).json()

# Look at top-level keys
print(glossary.keys())

import json
print(json.dumps(glossary.get("items", {}), indent=2)[:1000])

dict_keys(['items', 'success', 'query', 'user', 'meta'])
{
  "stats_1272": {
    "label": "% of Team Three-Point Attempts",
    "abbrev": "%ThreeFA",
    "code": "pct3fa",
    "type": "player"
  },
  "stats_1273": {
    "label": "% of Team Three-Pointers",
    "abbrev": "%ThreeFM",
    "code": "pct3fm",
    "type": "player"
  },
  "stats_1274": {
    "label": "% of Team Assists",
    "abbrev": "%AST",
    "code": "pctast",
    "type": "player"
  },
  "stats_1275": {
    "label": "% of Team Blocks",
    "abbrev": "%BLK",
    "code": "pctblk",
    "type": "player"
  },
  "stats_1276": {
    "label": "% of Team Fouls",
    "abbrev": "%F",
    "code": "pctf",
    "type": "player"
  },
  "stats_1277": {
    "label": "% of Team Shot Attempts",
    "abbrev": "%FGA",
    "code": "pctfga",
    "type": "player"
  },
  "stats_1278": {
    "label": "% of Team Field Goals",
    "abbrev": "%FGM",
    "code": "pctfgm",
    "type": "player"
  },
  "stats_1279": {
    "label": "% of Team Free Throw Att

In [6]:
import requests
import pandas as pd

# ================================
# NatStat API Configuration
# ================================
API_KEY = "7f10-3c2730"
BASE_URL = "https://api3.natst.at"
SPORT_CODE = "MBB"  # NCAA Men's Division I Basketball
SEASON = "2024"      # Season to query

# ================================
# Helper Functions
# ================================

def fetch_json(url):
    """Helper to fetch JSON from a URL."""
    response = requests.get(url)
    response.raise_for_status()
    return response.json()

def get_team_stat_codes():
    """Fetch glossary and return a list of valid team stat codes."""
    glossary_url = f"{BASE_URL}/{API_KEY}/glossary/{SPORT_CODE}/{SEASON}"
    glossary_json = fetch_json(glossary_url)

    items = glossary_json.get("items", {})
    team_stats = {v["code"]: v["label"] for k, v in items.items() if v["type"] == "team"}

    return team_stats

def fetch_team_stat(stat_code):
    """Fetch a team stat table for the season."""
    url = f"{BASE_URL}/{API_KEY}/stats/{SPORT_CODE}/team,{stat_code},{SEASON}"
    data = fetch_json(url)

    records = data.get("items", []) or data.get("stats", [])

    if not records:
        print(f"No data for stat: {stat_code}")
        return None

    # Flatten JSON to DataFrame
    df = pd.json_normalize(records)

    # Ensure team identifiers exist
    if "team_code" not in df.columns and "code" in df.columns:
        df.rename(columns={"code": "team_code", "name": "team_name"}, inplace=True)

    df["stat_code"] = stat_code
    return df

# ================================
# Fetch all team stats
# ================================

team_stat_codes = get_team_stat_codes()
print(f"Found {len(team_stat_codes)} team stat codes:", list(team_stat_codes.keys())[:10])

team_stats_dfs = {}

for stat_code in team_stat_codes:
    print(f"Fetching {stat_code}...")
    df = fetch_team_stat(stat_code)
    if df is not None:
        team_stats_dfs[stat_code] = df

print("\nFetched team stats DataFrames:", list(team_stats_dfs.keys())[:10])

# Example: access points DataFrame (if available)
# points_df = team_stats_dfs.get("pts")
# print(points_df.head())

Found 74 team stat codes: ['2fa', '2fa-d', '2fgpct', '2fgpct-d', '2fm', '2fm-d', '3fa', '3fa-d', '3fgpct', '3fgpct-d']
Fetching 2fa...
Fetching 2fa-d...
Fetching 2fgpct...
No data for stat: 2fgpct
Fetching 2fgpct-d...
No data for stat: 2fgpct-d
Fetching 2fm...
Fetching 2fm-d...
Fetching 3fa...
Fetching 3fa-d...
Fetching 3fgpct...
No data for stat: 3fgpct
Fetching 3fgpct-d...
No data for stat: 3fgpct-d
Fetching 3fm...
Fetching 3fm-d...
Fetching apg...
Fetching ast...
Fetching ast-d...
Fetching blk...
Fetching blk-d...
Fetching bpg...
Fetching confl...
Fetching confpct...
Fetching confw...
Fetching d-ppp...
Fetching drbpct...
No data for stat: drbpct
Fetching dreb...
Fetching dreb-d...
Fetching f...
Fetching f-d...
Fetching fgpct...
No data for stat: fgpct
Fetching fgpct-d...
No data for stat: fgpct-d
Fetching fga...
Fetching fga-d...
Fetching fgm...
Fetching fgm-d...
Fetching fpg...
Fetching ftpct...
No data for stat: ftpct
Fetching ftpct-d...
No data for stat: ftpct-d
Fetching ftfga...

In [7]:
print(team_stats_dfs.keys())
ftm_df = team_stats_dfs.get("ftm")
print(ftm_df.head())

dict_keys(['2fa', '2fa-d', '2fm', '2fm-d', '3fa', '3fa-d', '3fm', '3fm-d', 'apg', 'ast', 'ast-d', 'blk', 'blk-d', 'bpg', 'confl', 'confpct', 'confw', 'd-ppp', 'dreb', 'dreb-d', 'f', 'f-d', 'fga', 'fga-d', 'fgm', 'fgm-d', 'fpg', 'fta', 'fta-d', 'ftm', 'ftm-d', 'g', 'l', 'min', 'min-d', 'o-ppp', 'oreb', 'oreb-d', 'pa', 'pf', 'pace', 'poss', 'reb', 'reb-d', 'rpg', 'rpi', 'sos', 'spg', 'stl', 'stl-d', 'to', 'to-d', 'topg', 'w', 'winpct'])
  stat          statname season  stat_622434852.team stat_622434852.team-id  \
0  FTM  Free Throws Made   2024  High Point Panthers                    544   

  stat_622434852.team-code stat_622434852.value stat_622434852.ranking  \
0                      HPU                  722                      1   

  stat_622434852.time-stamp     stat_622433502.team  ... stat_622447452.value  \
0       2024-03-27 20:24:33  Grand Canyon Antelopes  ...                  500   

  stat_622447452.ranking stat_622447452.time-stamp stat_622443102.team  \
0               

In [8]:
import pandas as pd

def reshape_team_stat(df, stat_code):
    """
    Reshape a single team stat DataFrame from wide to long.

    Parameters:
        df (pd.DataFrame): raw stat DataFrame from API
        stat_code (str): the stat code, e.g., 'ftm', 'fgm'

    Returns:
        pd.DataFrame: long-format DataFrame with columns:
                      ['team_code', 'team_name', stat_code]
    """
    # Select all columns that contain '.team' or '.team-code' or '.value'
    team_cols = [col for col in df.columns if '.team' in col or '.team-code' in col or '.value' in col]
    long_rows = []

    for col in team_cols:
        if col.endswith('.team'):
            base = col.split('.')[0]
            for idx, team_name in enumerate(df[col]):
                team_code = df.get(f"{base}.team-code", [None]*len(df))[idx]
                value = df.get(f"{base}.value", [None]*len(df))[idx]
                long_rows.append({
                    "team_name": team_name,
                    "team_code": team_code,
                    stat_code: value
                })

    return pd.DataFrame(long_rows)

# Example: reshape FTM
ftm_df = team_stats_dfs.get("ftm")
ftm_long = reshape_team_stat(ftm_df, "ftm")
print(ftm_long.head())

# Now you can safely merge multiple stats by 'team_code' and 'team_name'

                  team_name team_code  ftm
0       High Point Panthers       HPU  722
1    Grand Canyon Antelopes       GCU  676
2       Purdue Boilermakers       PUR  675
3  Alabama A&amp;M Bulldogs      ALAM  654
4      Alabama Crimson Tide       ALA  650


In [9]:
# ----------------------------
# Combine all team stats into a single DataFrame
# ----------------------------

combined_team_stats = None

for stat_code, stat_df in team_stats_dfs.items():
    # Reshape the individual stat DataFrame (pass stat_code too)
    reshaped_df = reshape_team_stat(stat_df, stat_code)

    if reshaped_df is None or reshaped_df.empty:
        continue  # Skip if there is no data

    # Merge into combined DataFrame
    if combined_team_stats is None:
        combined_team_stats = reshaped_df
    else:
        combined_team_stats = pd.merge(
            combined_team_stats,
            reshaped_df,
            on=["team_name", "team_code"],
            how="outer"
        )

# Inspect the final merged DataFrame
print("Combined Team Stats shape:", combined_team_stats.shape)
print(combined_team_stats.head())

Combined Team Stats shape: (410, 57)
                    team_name team_code   2fa 2fa-d  2fm 2fm-d   3fa 3fa-d  \
0  Abilene Christian Wildcats       ACU  1457  1311  681   689   NaN   NaN   
1           Air Force Falcons       AFA   NaN   NaN  NaN   NaN   NaN   NaN   
2                  Akron Zips       AKR   NaN  1246  NaN   629   862   NaN   
3    Alabama A&amp;M Bulldogs      ALAM  1402  1270  NaN   NaN   NaN   NaN   
4        Alabama Crimson Tide       ALA   NaN  1476  723   751  1108   873   

   3fm 3fm-d  ...   rpi   sos  spg  stl stl-d   to to-d  topg    w winpct  
0  NaN   NaN  ...  None  None  7.7  262   227  419  472  12.3  NaN    NaN  
1  NaN   NaN  ...  None  None  NaN  NaN   NaN  NaN  NaN   NaN  NaN    NaN  
2  280   NaN  ...  None  None  NaN  NaN   NaN  NaN  NaN   NaN   24   .706  
3  NaN   261  ...  None  None  7.5  261   331  535  463  15.3  NaN    NaN  
4  413   277  ...  None  None  NaN  256   273  429  401   NaN  NaN   .656  

[5 rows x 57 columns]


#### ELO Ratings

In [10]:
elo_records = fetch_paginated("elo", SEASON)
elo_df = pd.json_normalize(elo_records)

print("ELO DF shape:", elo_df.shape)
elo_df.head()

ELO DF shape: (362, 244422)


,team,code,elo,elorank,lastgame.game_1484185.id,lastgame.game_1484185.gameday,lastgame.game_1484185.visitor-code,lastgame.game_1484185.visitor,lastgame.game_1484185.scorevis,lastgame.game_1484185.home-code,...,lastgame.game_1487138.gamestatus,lastgame.game_1487138.winorloss,lastgame.game_1487138.elo-beforegame,lastgame.game_1487138.elo-change,lastgame.game_1487138.elo-kfactor,lastgame.game_1487138.elo-adjust,lastgame.game_1487138.elo-winexpectancy,lastgame.game_1487138.forecast-favorite-code,lastgame.game_1487138.forecast-favorite,lastgame.game_1487138.forecast-correct
0,Duke,DUKE,2135,1,1484185,2026-03-02,DUKE,Duke,93,NCSU,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Florida,FLA,2088,2,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Michigan,MICH,2064,3,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Houston,HOU,2058,4,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Arizona,ARIZ,2056,5,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### Save Data

In [11]:
# Save
# teams_df.to_parquet("teams.parquet", index=False)
# games_df.to_parquet("games.parquet", index=False)
# combined_team_stats.to_parquet("team_stats.parquet", index=False)
# elo_df.to_parquet("elo.parquet", index=False)

# Load later
teams_df = pd.read_parquet("teams.parquet")
games_df = pd.read_parquet("games.parquet")
combined_team_stats = pd.read_parquet("team_stats.parquet")
elo_df = pd.read_parquet("elo.parquet")

FileNotFoundError: [Errno 2] No such file or directory: 'teams.parquet'

#### Basic Structure Checks

In [ ]:
dfs = {
    "teams_df": teams_df,
    "games_df": games_df,
    "combined_team_stats": combined_team_stats,
    "elo_df": elo_df
}

for name, df in dfs.items():
    print(f"\n===== {name} =====")
    print("Shape:", df.shape)
    print("\nColumns:")
    print(df.columns.tolist())

#### Data Types & Memory

In [ ]:
for name, df in dfs.items():
    print(f"\n===== {name} =====")
    print(df.dtypes)
    print("\nMemory Usage (MB):")
    print(round(df.memory_usage(deep=True).sum() / 1_000_000, 2))

#### Missing Values

In [ ]:
for name, df in dfs.items():
    print(f"\n===== {name} =====")
    missing = df.isna().sum()
    missing = missing[missing > 0].sort_values(ascending=False)

    if len(missing) == 0:
        print("No missing values.")
    else:
        print(missing.head(20))  # show top 20

#### Duplicate Rows

In [ ]:
for name, df in dfs.items():
    dup_count = df.duplicated().sum()
    print(f"{name}: {dup_count} duplicate rows")

#### Key Integrity Checks

In [ ]:
print("Teams unique codes:", teams_df["code"].nunique(), "of", len(teams_df))
print("Teams unique names:", teams_df["name"].nunique())

print("\nGames unique IDs:", games_df["id"].nunique(), "of", len(games_df))

print("\nTeam stats unique team codes:", combined_team_stats["team_code"].nunique())

print("\nELO unique team codes:", elo_df["code"].nunique())

#### Numeric Summary

In [ ]:
for name, df in dfs.items():
    print(f"\n===== {name} =====")
    numeric_cols = df.select_dtypes(include="number")

    if numeric_cols.shape[1] == 0:
        print("No numeric columns.")
    else:
        print(numeric_cols.describe().T.head(15))

#### Convert Numeric Columns

In [ ]:
import numpy as np

# Convert games scores
games_df["score-vis"] = pd.to_numeric(games_df["score-vis"], errors="coerce")
games_df["score-home"] = pd.to_numeric(games_df["score-home"], errors="coerce")

# Convert date
games_df["gameday"] = pd.to_datetime(games_df["gameday"], errors="coerce")

# Convert team stats numeric columns
stat_cols = combined_team_stats.columns.difference(["team_name", "team_code"])

for col in stat_cols:
    combined_team_stats[col] = pd.to_numeric(combined_team_stats[col], errors="coerce")

# Convert ELO key fields
elo_df["elo"] = pd.to_numeric(elo_df["elo"], errors="coerce")
elo_df["elorank"] = pd.to_numeric(elo_df["elorank"], errors="coerce")

print("Conversion complete.")

#### Re-check data

In [ ]:
combined_team_stats.select_dtypes(include="number").head()

#### Clean winpct

In [ ]:
combined_team_stats["winpct"] = (
    combined_team_stats["winpct"]
    .astype(str)
    .str.replace(".", "0.", regex=False)
)

combined_team_stats["winpct"] = pd.to_numeric(
    combined_team_stats["winpct"], errors="coerce"
)

combined_team_stats["winpct"].describe()

#### Remove Duplicate Games

In [ ]:
print("Before:", games_df.shape)

games_df = games_df.drop_duplicates(subset="id")

print("After:", games_df.shape)

#### Remove Duplicate Team Stats

In [ ]:
print("Before:", combined_team_stats.shape)

combined_team_stats = combined_team_stats.drop_duplicates(subset="team_code")

print("After:", combined_team_stats.shape)

#### Create Home Win Target

In [ ]:
games_df = games_df.copy()

games_df["home_win"] = np.where(
    games_df["score-home"] > games_df["score-vis"], 1, 0
)

games_df["point_diff"] = games_df["score-home"] - games_df["score-vis"]

games_df[["score-home", "score-vis", "home_win", "point_diff"]].head()

#### Distribution Checks

In [ ]:
import matplotlib.pyplot as plt

# Score distribution
games_df["score-home"].hist(bins=30)
plt.title("Home Score Distribution")
plt.show()

# ELO distribution
elo_df["elo"].hist(bins=30)
plt.title("ELO Distribution")
plt.show()

# Win % distribution
combined_team_stats["winpct"].hist(bins=30)
plt.title("Win Percentage Distribution")
plt.show()

- Home score and ELO both roughly bell shaped

#### Reduce ELO Dataset

In [ ]:
elo_model_df = elo_df[["team", "code", "elo", "elorank"]].copy()

print("Original shape:", elo_df.shape)
print("Reduced shape:", elo_model_df.shape)

#### Missingness Percentage

In [ ]:
missing_pct = (
    combined_team_stats
    .isna()
    .mean()
    .sort_values(ascending=False)
)

missing_summary = pd.DataFrame({
    "missing_pct": missing_pct,
    "non_null_count": combined_team_stats.notna().sum()
})

print(missing_summary.head(25))

#### Drop Sparse Features

In [ ]:
threshold = 0.5  # drop columns with >50% missing

cols_to_keep = missing_pct[missing_pct < threshold].index
combined_team_stats_clean = combined_team_stats[cols_to_keep]

print("Original columns:", combined_team_stats.shape[1])
print("Remaining columns:", combined_team_stats_clean.shape[1])

#### Drop Rows With Too Many Missing Values

In [ ]:
row_missing_pct = combined_team_stats_clean.isna().mean(axis=1)

combined_team_stats_clean = combined_team_stats_clean[row_missing_pct < 0.6]

print("Remaining teams:", combined_team_stats_clean.shape[0])

#### Feature Distribution Summary

In [ ]:
numeric_stats = combined_team_stats.select_dtypes(include="number")

summary = numeric_stats.describe().T.sort_values("std", ascending=False)

print(summary.head(20))

#### Correlation Matrix

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12,10))
sns.heatmap(
    numeric_stats.corr(),
    cmap="coolwarm",
    center=0
)
plt.title("Team Stats Correlation Matrix")
plt.show()

- `winpct` looks positively correlated with:
    - rebounding
    - steals
- Negatively correlated with:
    - turnovers (`to`, `topg`)
 - Tight clusters among multiple features - will need to reduce. Ex:
     - `fga`, `fgm`
     - `2fa`, `2fm`
     - `3fa`, `3fm`
     - `fta-d`, `ftm-d`
     - `reb`, `rpg`
     - `oreb`, `reb`
     - `pa`, `pace`, `poss`

#### Merge ELO + Win %

In [ ]:
elo_model_df = elo_df[["code", "elo", "elorank"]].copy()

team_model_check = combined_team_stats.merge(
    elo_model_df,
    left_on="team_code",
    right_on="code",
    how="inner"
)

team_model_check[["elo", "winpct"]].corr()

- Hm... this seems low. Will need to investigate structure

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(team_model_check["elo"], team_model_check["winpct"])
plt.xlabel("ELO")
plt.ylabel("Win %")
plt.title("ELO vs Win %")
plt.show()

#### Merge Example (Team Stats + ELO)

In [ ]:
# Attempt merge if common team identifier exists
common_cols = list(set(team_stats_df.columns).intersection(set(elo_df.columns)))

if len(common_cols) > 0:
    merge_key = common_cols[0]
    merged_df = pd.merge(team_stats_df, elo_df, on=merge_key, how="inner")
    print(f"\nMerged Dataset Shape (Team Stats + ELO): {merged_df.shape}")

#### Home Win Distribution

In [ ]:
games_df["home_win"].value_counts(normalize=True)

## Feature Engineering

## Model Building